# Feature Engineering + Weighted K-Means

Build four banking behavior dimensions, standardize them, test K, and profile segments.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
df=pd.read_csv(Path("../data/customer_data_clean.csv"))

## 1. Financial capacity

In [ ]:
def pct_rank(s): return s.rank(pct=True,method="average")
df["financial_capacity"]=0.55*pct_rank(np.log1p(df.annual_income))+0.30*pct_rank(np.log1p(df.account_balance))+0.15*pct_rank(np.log1p(df.investment_amount))

## 2. Engagement

In [ ]:
df["engagement"]=0.45*pct_rank(np.log1p(df.monthly_transaction_count))+0.25*pct_rank(np.log1p(df.monthly_transaction_amount))+0.15*df.digital_transaction_ratio+0.10*(1-pct_rank(df.months_inactive_12_mon))+0.05*pct_rank(df.total_relationship_count)

## 3. Credit dependence

In [ ]:
df["credit_dependence"]=0.40*pct_rank(df.credit_utilization_ratio)+0.35*pct_rank(np.log1p(df.loan_amount))+0.25*pct_rank(df.emi_to_monthly_income_ratio.clip(0,3))

## 4. Digital behavior

In [ ]:
df["digital_behavior"]=0.70*df.digital_transaction_ratio+0.30*df.upi_transaction_share

## 5. Standardize + unequal business weights

In [ ]:
cols=["financial_capacity","engagement","credit_dependence","digital_behavior"]
X=StandardScaler().fit_transform(df[cols])
X=X*np.sqrt(np.array([0.35,0.30,0.20,0.15]))

## 6. Compare K=2..6

In [ ]:
rng=np.random.default_rng(42); sample_idx=rng.choice(len(df),12000,replace=False)
rows=[]
for k in range(2,7):
    km=KMeans(n_clusters=k,n_init=15,random_state=42); lab=km.fit_predict(X)
    rows.append({"K":k,"Inertia":km.inertia_,"Silhouette":silhouette_score(X[sample_idx],lab[sample_idx]),"Calinski_Harabasz":calinski_harabasz_score(X,lab),"Davies_Bouldin":davies_bouldin_score(X,lab)})
selection=pd.DataFrame(rows); selection

## 7. Fit K=4 and profile

In [ ]:
model=KMeans(n_clusters=4,n_init=20,random_state=42)
df["segment_id"]=model.fit_predict(X)+1
df.groupby("segment_id")[cols].mean().round(3)

## 8. Business interpretation

The final observed segments are named after profiling: Prime Engaged, Digital Active, Credit Active, Affluent Under-Engaged.

In [ ]:
pca=PCA(n_components=2,random_state=42).fit_transform(X)
plt.figure(figsize=(8,6)); plt.scatter(pca[sample_idx,0],pca[sample_idx,1],c=df.segment_id.to_numpy()[sample_idx],s=8,alpha=.45); plt.title("Customer Segments — PCA View"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()